# arc3-flashnext-smoke — STOCK duck × Flash-Next NVFP4 (the model-swap read)

One phase, all 25 public games, eval geometry (7,920 s/game, concurrency 28
from the serialized solver), served by sonpham's **Qwen3.8-Flash-Next NVFP4**
package on ONE RTX Pro 6000 (gate-proven boot: 740 s on rung `gcp_exact`,
`submission/_flashnext_gate/results_v4/`). The harness is the **stock duck**
(anim-20260807 bundle): no grafts, `ONLY_RESET_LEVELS=true`, stock sampling
(temp 0.6 / top-p 0.95 / top-k 20).

**Deviation vs the 27B baseline:** `LOCAL_ANALYZER_CONTEXT_WINDOW=24576` +
`LOCAL_ANALYZER_MAX_OUTPUT=4096`, because their server serves
`--max-model-len 32768` (the 27B ran a 65536-ctx server with a 32768 window
and no max_tokens). Fair enough: the stock 27B's effective history is 4-9
turns anyway.

**Pre-registered read** — baseline = pooled stock 27B (3 kernels: levels/game
1.00 / 1.04 / 0.84-0.88, zero-level 8-9/25, mean local score 3.5-4.9):

* **PASS** = flashnext levels/game >= 1.3 OR zero-level <= 6 with levels >= 1.0
* **INCONCLUSIVE** = levels/game 0.9-1.3
* **FAIL** = levels/game < 0.9

Also record per-game actions (expect MORE actions/game from the faster
serving: gate conc-28 measured 412 vs the 27B's 297 gen tok/s aggregate) and
the server queue (28 workers vs `--max-num-seqs 22` — requests queue by
design; running/waiting/kv sampled every 60 s).


In [ ]:
import contextlib
import json
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import TextIO
from urllib.request import urlopen


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


NOTEBOOK_START_EPOCH = time.time()
RUN_AS_SUBMISSION = False
RUN_AS_SUBMISSION = RUN_AS_SUBMISSION or _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
ENABLE_GPU = True

os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if RUN_AS_SUBMISSION else "0"
os.environ.setdefault("MPLBACKEND", "Agg")

if ENABLE_GPU:
    cuda_library_path = "/usr/local/nvidia/lib64"
    existing = [entry for entry in os.environ.get("LIBRARY_PATH", "").split(os.pathsep) if entry]
    os.environ["LIBRARY_PATH"] = os.pathsep.join(
        [cuda_library_path, *[entry for entry in existing if entry != cuda_library_path]]
    )

print(f"TAAF RUN_AS_SUBMISSION={RUN_AS_SUBMISSION}")
if ENABLE_GPU:
    print(f"taaf.kaggle: LIBRARY_PATH={os.environ['LIBRARY_PATH']}")

In [ ]:
# Fail-fast GPU assert: metadata machine_shape + --accelerator alone can still
# bind P100; the competition source attachment is the real RTX Pro 6000 gate.
import subprocess as _sp

_gpu = _sp.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("boot gpu:", (_gpu.stdout or "").strip() or (_gpu.stderr or "").strip())
_gpu_name = (_gpu.stdout or "").upper()
assert "RTX" in _gpu_name and "6000" in _gpu_name, (
    f"GPU misbind — expected RTX Pro 6000, got: {_gpu.stdout!r} {_gpu.stderr!r}"
)


In [ ]:
wheelhouse = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")
if wheelhouse.exists():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--no-warn-conflicts",
            "--disable-pip-version-check",
            "--find-links",
            str(wheelhouse),
            "arc-agi",
        ]
    )
elif os.getenv("TAAF_KAGGLE_BUNDLE_DIR"):
    print(f"Competition wheelhouse not found at {wheelhouse}; assuming local debug dependencies are installed.")
else:
    raise RuntimeError(f"Competition wheelhouse not found at {wheelhouse}.")

In [ ]:
# ================= flashnext-smoke config — bundle, sys.path, results sink =================
# REPLACES the v12 "Qwen3.8 / Kaggle input configuration" cell. The 27B Kaggle
# Model is deliberately NOT attached: the model axis IS the experiment. The
# serving stack (sonpham's Flash-Next NVFP4 package) is assembled below.
import re
import shutil
import signal
import traceback
import urllib.request

WORKING_DIR = Path(os.getenv("TAAF_KAGGLE_WORKING_DIR", "/kaggle/working")).resolve()
WORKING_DIR.mkdir(parents=True, exist_ok=True)
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
SOFT_DEADLINE_BUFFER_S = 600.0
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"

# Keep the whole run offline. vLLM/Transformers must use the mounted files only.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
os.environ["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)


def _find_taaf_bundle() -> Path:
    explicit = os.getenv("TAAF_KAGGLE_BUNDLE_DIR", "").strip()
    # v2: sonpham's flashnext datasets ALSO contain a taaf-kaggle-bundle.json
    # (their apex fork). v1 rglobbed /kaggle/input and picked theirs first ->
    # their solver source + our pickle -> AttributeError hard_noop_guard.
    # Pin to the anim-20260807 bundle (the bytes the pickle was built from).
    for pinned in (Path("/kaggle/input/taaf-kaggle-source-anim-20260807-anim"),
                   Path("/kaggle/input/datasets/jakobbrggen/taaf-kaggle-source-anim-20260807-anim")):
        if (pinned / DATASET_BUNDLE_MARKER).is_file():
            return pinned
    if explicit and (Path(explicit) / DATASET_BUNDLE_MARKER).is_file():
        return Path(explicit)
    for root in [Path("/kaggle/input/datasets"), Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            for marker in root.rglob(DATASET_BUNDLE_MARKER):
                return marker.parent
    raise RuntimeError("Could not find TAAF Kaggle source bundle dataset.")


BUNDLE_DIR = _find_taaf_bundle()
os.environ["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
print(f"TAAF source bundle: {BUNDLE_DIR}")


# Make bundled TAAF repos importable for this notebook and child Python
# processes (verbatim from the v12 setup cell — the part we keep).
def _source_path_entries(bundle_dir: Path) -> list[Path]:
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        return []
    entries: list[Path] = []
    for repo in sorted(src_root.iterdir(), reverse=True):
        if not repo.is_dir():
            continue
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))
if source_entries:
    import sysconfig

    pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
    pth_path.write_text("".join(f"{entry}\n" for entry in source_entries), encoding="utf-8")
    print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)", flush=True)

# ---- boot-results sink (flashnext-gate pattern): partial state always lands on disk ----
RESULTS_PATH = WORKING_DIR / "flashnext_smoke_boot.json"
RESULTS = {
    "meta": {"kernel": "arc3-flashnext-smoke",
             "started_utc": datetime.utcnow().isoformat() + "Z"},
    "boots": {},
    "phases": {},
    "verdicts": {},
}


def elapsed_min():
    return (time.time() - NOTEBOOK_START_EPOCH) / 60.0


def save_results():
    tmp = RESULTS_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(RESULTS, indent=2, default=str) + "\n", encoding="utf-8")
    tmp.replace(RESULTS_PATH)


def host_snapshot(tag):
    snap = {}
    try:
        mem = dict((ln.split(":", 1)[0], ln.split(":", 1)[1].strip())
                   for ln in Path("/proc/meminfo").read_text().splitlines() if ":" in ln)
        snap["mem_total"] = mem.get("MemTotal")
        snap["mem_available"] = mem.get("MemAvailable")
    except Exception as exc:
        snap["meminfo_error"] = repr(exc)[:120]
    for mount in ("/kaggle/working", "/kaggle/tmp", "/tmp", "/"):
        try:
            usage = shutil.disk_usage(mount)
            snap[mount] = f"free {usage.free / 1e9:.1f} / total {usage.total / 1e9:.1f} GB"
        except Exception:
            snap[mount] = "n/a"
    RESULTS["meta"].setdefault("host", {})[tag] = snap
    return snap


print("flashnext-smoke: host snapshot", json.dumps(host_snapshot("start"), indent=1), flush=True)
save_results()


In [ ]:
# Audit the attached inputs that matter for this run. NOTE: no 27B model
# mount — the Kaggle models input dir is expected to be ABSENT.
INPUT_ROOT = Path("/kaggle/input")
print("=== TAAF bundle ===")
print(BUNDLE_DIR, "exists:", BUNDLE_DIR.exists())
for _needle in ("serving-part-000", "serving-part-001", "serving-part-002"):
    _hits = sorted({str(p) for p in INPUT_ROOT.rglob(_needle) if p.is_dir()})
    print(f"=== {_needle} ===", _hits)
_tarballs = sorted({str(p) for p in INPUT_ROOT.rglob("flashnext-gcp-container-site-packages.tar.zst")})
print("=== runtime tarball ===", _tarballs)
_wheels = [str(p) for p in (INPUT_ROOT / "arc3-qwen36-runtime-wheels",
                            INPUT_ROOT / "datasets" / "jcole75" / "arc3-qwen36-runtime-wheels")
           if p.is_dir()]
print("=== cu13 wheelhouse ===", _wheels)
_envdirs = [str(p) for p in (
    Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"),
    Path("/kaggle/input/arc-prize-2026-arc-agi-3/environment_files")) if p.is_dir()]
print("=== competition environment_files ===", _envdirs)
print("=== input models dir (should NOT exist) ===", (INPUT_ROOT / "models").exists())
_smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print((_smi.stdout or "").strip()[:900], flush=True)


In [ ]:
# ================= Phase 1 — ASSEMBLE: their runtime + symlink-union model view =================
# Pins below are copied verbatim from sonpham's kaggle_flashnext_setup.py
# (part-A source-bundle). Their preconverted wrapper verifies a PRIVATE Kaggle
# MODEL mount, so we assemble the identical flat view from the three public
# serving-part datasets and reuse everything else of their serve chain.
EXPECTED_RUNTIME_SHA256 = "c06a78d59a74ac278dc2278d26dde6c70c48a4e28bb91fd4fbbefff4484e10f3"
EXPECTED_ZSTD_SHA256 = "7c5468b370f7c47eda07281e3437fafc568f95d10420051e3aa522709f9342c5"
EXPECTED_VERSIONS_LINE = "0.1.dev20073+g8e685d198 2.13.0+cu130 5.15.1 13.0"
RUNTIME_ARCHIVE = "flashnext-gcp-container-site-packages.tar.zst"
QWEN_SERVED_MODEL_NAME = "RadixArk/Qwen3.8-Flash-Next-NVFP4"
MODEL_REVISION = "7b719225242aacd3dbd3f9407468c2ee9a9d2594"
INPUT_ROOT = Path("/kaggle/input")
CU13_HOME = globals().get("CU13_HOME") or {"value": None}

# scratch root: the extracted runtime is ~15 GB — keep it OFF /kaggle/working
# (its ~20 GB doubles as the preserved-output volume).
def _pick_scratch():
    best, best_free = Path("/tmp"), 0
    for cand in (Path("/kaggle/tmp"), Path("/kaggle/temp"), Path("/tmp")):
        try:
            cand.mkdir(parents=True, exist_ok=True)
            free = shutil.disk_usage(str(cand)).free
        except Exception:
            continue
        if free > best_free:
            best, best_free = cand, free
    return best, best_free


SCRATCH_ROOT, _scratch_free = _pick_scratch()
RUNTIME_ROOT = SCRATCH_ROOT / "flashnext-gcp-runtime"
SITE_PACKAGES = RUNTIME_ROOT / "dist-packages"
MODEL_DIR = SCRATCH_ROOT / "flashnext-model"
QWEN_MODEL_PATH = MODEL_DIR
print(f"flashnext-smoke: scratch root {SCRATCH_ROOT} (free {_scratch_free / 1e9:.1f} GB)", flush=True)
RESULTS["meta"]["scratch_root"] = str(SCRATCH_ROOT)


def sha256_file(path):
    digest = __import__("hashlib").sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(16 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def serving_env():
    # VERBATIM from their kaggle_flashnext_setup.py serving_env() — including
    # VLLM_PLE_CPU_OFFLOAD=1 (the single-GPU mechanism: ~104 GB of PLE n-gram
    # tables live in host RAM) and TORCH_CUDA_ARCH_LIST=12.0f (Blackwell).
    env = os.environ.copy()
    current_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = (
        str(SITE_PACKAGES)
        if not current_pythonpath
        else f"{SITE_PACKAGES}{os.pathsep}{current_pythonpath}"
    )
    packaged_library_dirs = sorted(
        str(path) for path in (SITE_PACKAGES / "nvidia").glob("*/lib") if path.is_dir()
    )
    system_library_dirs = [
        "/usr/local/nvidia/lib64",
        "/usr/local/cuda/lib64",
        "/usr/local/nvidia/lib",
    ]
    current_ld = [entry for entry in env.get("LD_LIBRARY_PATH", "").split(os.pathsep) if entry]
    env["LD_LIBRARY_PATH"] = os.pathsep.join(
        dict.fromkeys(packaged_library_dirs + system_library_dirs + current_ld)
    )
    current_path = [entry for entry in env.get("PATH", "").split(os.pathsep) if entry]
    env["PATH"] = os.pathsep.join(
        dict.fromkeys(["/usr/local/nvidia/bin", "/usr/local/cuda/bin"] + current_path)
    )
    env.update(
        {
            "USE_TF": "0",
            "TRANSFORMERS_NO_TF": "1",
            "TRANSFORMERS_NO_TORCHVISION": "1",
            "VLLM_NO_USAGE_STATS": "1",
            "VLLM_ENABLE_CUDA_COMPATIBILITY": "0",
            "VLLM_PLE_CPU_OFFLOAD": "1",
            "VLLM_PLE_OFFLOAD_READY_TIMEOUT": "1800",
            "PYTORCH_ALLOC_CONF": "expandable_segments:False",  # v2: True needs pidfd_getfd CUDA-IPC, blocked by Kaggle seccomp (v1 boot death in the PLE offload worker)
            "HF_HUB_OFFLINE": "1",
        }
    )
    # v3: "12.0f" made the tarball's flashinfer filter out every major-12 arch
    # ("No supported CUDA architectures found for major versions [12]" in the
    # sm120 fused-MoE JIT). Ladder now controls the arch env: None = unset
    # (flashinfer derives SM 12.0 from the device), or an explicit string.
    env.pop("TORCH_CUDA_ARCH_LIST", None)
    arch = ARCH_OVERRIDE.get("value")
    if arch:
        env["TORCH_CUDA_ARCH_LIST"] = arch
    # v4: flashinfer's sm120 fused-MoE JIT dropped every major-12 arch because
    # the nvcc it found (the image's old /usr/local/cuda) predates SM 12.0.
    # Point the whole toolchain at the CUDA-13.3 pip wheels installed at
    # assemble time (the jcole75 wheelhouse recipe: nvidia/cu13 layout).
    cu13 = CU13_HOME.get("value")
    if cu13:
        env["CUDA_HOME"] = cu13
        env["CUDA_PATH"] = cu13
        env["FLASHINFER_NVCC"] = str(Path(cu13) / "bin" / "nvcc")
        env["FLASHINFER_EXTRA_LDFLAGS"] = f"-L{Path(cu13) / 'lib'} -L/usr/local/nvidia/lib64"
        env["PATH"] = f"{Path(cu13) / 'bin'}{os.pathsep}" + env.get("PATH", "")
        env["LD_LIBRARY_PATH"] = f"{Path(cu13) / 'lib'}{os.pathsep}/usr/local/nvidia/lib64{os.pathsep}" + env.get("LD_LIBRARY_PATH", "")
        env["LIBRARY_PATH"] = f"{Path(cu13) / 'lib'}{os.pathsep}/usr/local/nvidia/lib64{os.pathsep}" + env.get("LIBRARY_PATH", "")
        env["CPATH"] = f"{Path(cu13) / 'include'}{os.pathsep}" + env.get("CPATH", "")
    return env


CU13_HOME = globals().get("CU13_HOME") or {"value": None}


ARCH_OVERRIDE = {"value": None}


RUNTIME_OK = False
ASSEMBLE = {}
try:
    _t0 = time.time()
    # ---- locate mounts (any nesting: /kaggle/input/<slug> or /kaggle/input/datasets/<user>/<slug>)
    _shards = {}
    for _name in ("serving-part-000", "serving-part-001", "serving-part-002"):
        _hits = sorted({p.resolve() for p in INPUT_ROOT.rglob(_name) if p.is_dir()})
        if len(_hits) != 1:
            raise FileNotFoundError(f"expected exactly one mounted {_name}, got {_hits}")
        _shards[_name] = _hits[0]
    _bundles = sorted({p.parent.resolve() for p in INPUT_ROOT.rglob("source-bundle/zstd")})
    if len(_bundles) != 1:
        raise FileNotFoundError(f"expected exactly one source-bundle, got {_bundles}")
    BUNDLE_DIR = _bundles[0]
    _archives = sorted({p.resolve() for p in INPUT_ROOT.rglob(RUNTIME_ARCHIVE)})
    if not _archives:
        raise FileNotFoundError(f"runtime archive {RUNTIME_ARCHIVE} not mounted")
    _preferred = [p for p in _archives if "runtime-exact" in str(p)]
    RUNTIME_TARBALL = (_preferred or _archives)[0]
    ASSEMBLE["shard_dirs"] = {k: str(v) for k, v in _shards.items()}
    ASSEMBLE["bundle_dir"] = str(BUNDLE_DIR)
    ASSEMBLE["runtime_tarball"] = str(RUNTIME_TARBALL)
    print("flashnext-smoke: shards", ASSEMBLE["shard_dirs"], flush=True)

    # ---- v4: CUDA-13.3 compiler toolchain for the flashinfer sm120 JIT ----
    # The tarball may or may not carry an nvcc; the image's is too old for
    # SM 12.0. Prefer a cu13 nvcc found inside the extracted runtime, else
    # install the pinned CUDA-13.3 wheels from the jcole75 wheelhouse.
    def _wire_cu13():
        hits = sorted(RUNTIME_ROOT.rglob("nvidia/cu13/bin/nvcc")) if RUNTIME_ROOT.exists() else []
        if hits:
            CU13_HOME["value"] = str(hits[0].parent.parent)
            print("flashnext-smoke: cu13 nvcc found in runtime:", hits[0], flush=True)
            return
        wh = None
        for cand in (INPUT_ROOT / "arc3-qwen36-runtime-wheels",
                     INPUT_ROOT / "datasets" / "jcole75" / "arc3-qwen36-runtime-wheels"):
            if (cand / "wheels").is_dir():
                wh = cand / "wheels"
                break
        if wh is None:
            print("flashnext-smoke: WARNING no cu13 nvcc and no wheelhouse mount — JIT will fail", flush=True)
            return
        target = SCRATCH_ROOT / "cu13-site"
        target.mkdir(parents=True, exist_ok=True)
        cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--find-links", str(wh),
               "--target", str(target), "--no-deps", "--disable-pip-version-check", "--no-warn-conflicts",
               "nvidia-cuda-nvcc==13.3.73", "nvidia-cuda-crt==13.3.73", "nvidia-cuda-runtime==13.3.29",
               "nvidia-cuda-cccl==13.3.3.4.1", "nvidia-cuda-nvrtc==13.3.33", "nvidia-nvvm==13.3.73",
               "nvidia-curand==10.4.3.29", "nvidia-cublas==13.3.0.5"]
        print("flashnext-smoke: installing cu13 toolchain:", " ".join(cmd[-8:]), flush=True)
        subprocess.run(cmd, check=True)
        nvcc = target / "nvidia" / "cu13" / "bin" / "nvcc"
        if nvcc.is_file():
            CU13_HOME["value"] = str(nvcc.parent.parent)
            out = subprocess.run([str(nvcc), "--version"], capture_output=True, text=True)
            print("flashnext-smoke: cu13 nvcc:", (out.stdout or out.stderr).strip().splitlines()[-1], flush=True)
        else:
            print("flashnext-smoke: WARNING cu13 wheels installed but no bin/nvcc at", nvcc, flush=True)

    _wire_cu13()
    ASSEMBLE["cu13_home"] = CU13_HOME["value"]
    print("flashnext-smoke: bundle", BUNDLE_DIR, "| tarball", RUNTIME_TARBALL, flush=True)

    # ---- their runtime install, verbatim semantics (sha-pinned tarball + zstd) ----
    _sha = sha256_file(RUNTIME_TARBALL)
    if _sha != EXPECTED_RUNTIME_SHA256:
        raise RuntimeError(f"runtime archive drift: {_sha} != {EXPECTED_RUNTIME_SHA256}")
    _zstd_src = BUNDLE_DIR / "zstd"
    _zsha = sha256_file(_zstd_src)
    if _zsha != EXPECTED_ZSTD_SHA256:
        raise RuntimeError(f"bundled zstd drift: {_zsha} != {EXPECTED_ZSTD_SHA256}")
    _tools = SCRATCH_ROOT / "packaging-tools"
    _tools.mkdir(parents=True, exist_ok=True)
    ZSTD_TOOL = _tools / "zstd"
    shutil.copy2(_zstd_src, ZSTD_TOOL)
    ZSTD_TOOL.chmod(0o755)
    shutil.rmtree(RUNTIME_ROOT, ignore_errors=True)
    RUNTIME_ROOT.mkdir(parents=True)
    _tx = time.time()
    subprocess.run(["tar", f"--use-compress-program={ZSTD_TOOL}", "-xf",
                    str(RUNTIME_TARBALL), "-C", str(RUNTIME_ROOT)], check=True)
    ASSEMBLE["extract_s"] = round(time.time() - _tx, 1)
    if not (SITE_PACKAGES / "vllm").is_dir():
        raise RuntimeError(f"extracted runtime incomplete: {SITE_PACKAGES}")
    _probe = subprocess.run(
        [sys.executable, "-c",
         ("import platform,torch,transformers,vllm; "
          "print(platform.python_version(),vllm.__version__,torch.__version__,"
          "transformers.__version__,torch.version.cuda)")],
        env=serving_env(), capture_output=True, text=True)
    ASSEMBLE["runtime_import"] = (_probe.stdout or "").strip()[:200]
    print("flashnext-smoke: exact runtime import:", ASSEMBLE["runtime_import"], flush=True)
    if _probe.returncode:
        raise RuntimeError(f"exact runtime import failed:\n{(_probe.stderr or '')[-3000:]}")
    if EXPECTED_VERSIONS_LINE not in (_probe.stdout or ""):
        raise RuntimeError(f"exact serving versions drifted: {ASSEMBLE['runtime_import']}")

    # ---- symlink-union model view (their zero-copy layout, dataset-mount flavour) ----
    shutil.rmtree(MODEL_DIR, ignore_errors=True)
    MODEL_DIR.mkdir(parents=True)
    _seen = {}
    for _name in sorted(_shards):
        for _member in sorted(_shards[_name].iterdir()):
            if not _member.is_file():
                raise RuntimeError(f"unexpected nested entry in {_name}: {_member}")
            if _member.name in _seen:
                raise RuntimeError(f"duplicate file across shards: {_member.name} "
                                   f"({_seen[_member.name]} vs {_name})")
            _seen[_member.name] = _name
            if _member.name == "model.safetensors.index.json":
                shutil.copy2(_member, MODEL_DIR / _member.name)
            else:
                (MODEL_DIR / _member.name).symlink_to(_member)
    _st_files = sorted(MODEL_DIR.glob("*.safetensors"))
    _st_bytes = sum(p.stat().st_size for p in _st_files)
    ASSEMBLE["model_files"] = len(_seen)
    ASSEMBLE["safetensor_files"] = len(_st_files)
    ASSEMBLE["safetensor_bytes"] = _st_bytes
    if len(_st_files) != 206:
        raise RuntimeError(f"expected 206 safetensors, got {len(_st_files)}")
    if _st_bytes < 186000000000:
        raise RuntimeError(f"payload too small: {_st_bytes}")
    _index = json.loads((MODEL_DIR / "model.safetensors.index.json").read_text())
    _needed = sorted(set((_index.get("weight_map") or {}).values()))
    _missing = [n for n in _needed if not (MODEL_DIR / n).is_file()]
    ASSEMBLE["index_files_referenced"] = len(_needed)
    if _missing:
        raise RuntimeError(f"index references missing files: {_missing[:8]} "
                           f"(+{max(0, len(_missing) - 8)} more)")
    if list(MODEL_DIR.glob("model-plefp8-*.safetensors")):
        raise RuntimeError("serving view still contains FP8 PLE source files")

    # ---- PLE-conversion provenance vs the GCP winner (their check, hashes-by-manifest) ----
    _expected = json.loads((BUNDLE_DIR / "FLASHNEXT_GCP_MODEL_INFO.json").read_text())["ple_conversion"]
    _observed = json.loads((MODEL_DIR / "ple-bf16-conversion.json").read_text())
    _exp_files = {i["target"]: (int(i["target_bytes"]), i["target_sha256"]) for i in _expected["files"]}
    _obs_files = {i["target"]: (int(i["target_bytes"]), i["target_sha256"]) for i in _observed["files"]}
    if _obs_files != _exp_files or _observed.get("index_sha256") != _expected.get("index_sha256"):
        raise RuntimeError("PLE conversion provenance differs from the GCP winner")
    for _tname, (_tbytes, _sha_unused) in _exp_files.items():
        if (MODEL_DIR / _tname).stat().st_size != _tbytes:
            raise RuntimeError(f"PLE file size drift: {_tname}")
    ASSEMBLE["ple_files_verified"] = len(_exp_files)
    _cfg = json.loads((MODEL_DIR / "config.json").read_text())
    ASSEMBLE["architectures"] = _cfg.get("architectures")
    ASSEMBLE["model_type"] = _cfg.get("model_type")
    ASSEMBLE["assemble_s"] = round(time.time() - _t0, 1)
    RUNTIME_OK = True
    print(f"flashnext-smoke: ASSEMBLE OK in {ASSEMBLE['assemble_s']} s — "
          f"{len(_st_files)} safetensors / {_st_bytes / 1e9:.1f} GB, "
          f"arch {ASSEMBLE['architectures']}, elapsed {elapsed_min():.1f} min", flush=True)
except Exception as _exc:
    traceback.print_exc()
    ASSEMBLE["error"] = repr(_exc)[:800]
    RESULTS["verdicts"]["boot"] = "ASSEMBLE-FAILED: " + repr(_exc)[:300]
    print("flashnext-smoke: ASSEMBLE FAILED — every later phase will be skipped", flush=True)
RESULTS["meta"]["assemble"] = ASSEMBLE
RESULTS["meta"]["model_path"] = str(QWEN_MODEL_PATH)
RESULTS["meta"]["served_model_name"] = QWEN_SERVED_MODEL_NAME
host_snapshot("after_assemble")
save_results()


In [ ]:
# ================= serve library — helpers + their launch argv (gate-proven) =================
# Minimal slice of the arc3-flashnext-gate serve chain: constants, HTTP/log/GPU
# helpers, and the server lifecycle (their serving_env + argv verbatim). The
# gate's load-generator/battery phases are NOT included — the load here is the
# real duck harness.
VLLM_HOST = "127.0.0.1"
VLLM_PORT = 1234
VLLM_ROOT = f"http://{VLLM_HOST}:{VLLM_PORT}"
VLLM_API = VLLM_ROOT + "/v1"
VLLM_MAX_MODEL_LEN = 32768   # their launch_server value

CURRENT_SERVER = {"proc": None, "log": str(WORKING_DIR / "vllm-gcp_exact.log"),
                  "tag": "none", "flags": []}


def http_json(url, payload=None, timeout=120):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))


def server_alive(timeout=5):
    try:
        http_json(VLLM_API + "/models", timeout=timeout)
        return True
    except Exception:
        return False


def vllm_procs():
    out = subprocess.run(["pgrep", "-f", "vllm.entrypoints"], capture_output=True, text=True)
    return [int(x) for x in out.stdout.split() if x.strip().isdigit()]


def gpu_sample():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=20)
        util, mem = out.stdout.strip().splitlines()[0].split(",")
        return {"util_pct": int(util.strip()), "mem_mib": int(mem.strip())}
    except Exception:
        return None


def tail_log_lines(path, max_bytes=524288):
    p = Path(path)
    if not p.exists():
        return []
    with p.open("rb") as handle:
        handle.seek(0, 2)
        size = handle.tell()
        handle.seek(max(0, size - max_bytes))
        return handle.read().decode("utf-8", errors="replace").splitlines()


# ---- their launch_server argv VERBATIM (kaggle_flashnext_setup.py, kv "auto") ----
FLASH_SERVE_FLAGS = [
    "--model", str(QWEN_MODEL_PATH),
    "--served-model-name", QWEN_SERVED_MODEL_NAME,
    "--host", VLLM_HOST,
    "--port", str(VLLM_PORT),
    "--tensor-parallel-size", "1",
    "--distributed-executor-backend", "mp",
    "--gpu-memory-utilization", "0.96",
    "--max-model-len", "32768",
    "--max-num-seqs", "22",
    "--max-num-batched-tokens", "6144",
    "--kv-cache-dtype", "auto",
    "--enable-prefix-caching",
    "--no-enable-flashinfer-autotune",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "qwen3_xml",
    "--generation-config", "vllm",
    "--default-chat-template-kwargs", '{"preserve_thinking": true}',
    "--reasoning-parser", "qwen3",
]
# pre-registered retry rung if their exact config fails (OOM or otherwise):
REDUCED_SERVE_FLAGS = [
    "--model", str(QWEN_MODEL_PATH),
    "--served-model-name", QWEN_SERVED_MODEL_NAME,
    "--host", VLLM_HOST,
    "--port", str(VLLM_PORT),
    "--tensor-parallel-size", "1",
    "--distributed-executor-backend", "mp",
    "--gpu-memory-utilization", "0.92",
    "--max-model-len", "16384",
    "--max-num-seqs", "12",
    "--max-num-batched-tokens", "6144",
    "--kv-cache-dtype", "auto",
    "--enable-prefix-caching",
    "--no-enable-flashinfer-autotune",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "qwen3_xml",
    "--generation-config", "vllm",
    "--default-chat-template-kwargs", '{"preserve_thinking": true}',
    "--reasoning-parser", "qwen3",
]
# (tag, flags, TORCH_CUDA_ARCH_LIST override): None = unset -> flashinfer
# derives SM 12.0 from the device; "12.0a" = the CUTLASS sm120a spelling.
BOOT_LADDER = [("gcp_exact", FLASH_SERVE_FLAGS, None),
               ("gcp_exact_arch120a", FLASH_SERVE_FLAGS, "12.0a"),
               ("reduced", REDUCED_SERVE_FLAGS, None)]


# ---- server lifecycle (their serving_env, own process group) --------------------
ENGINE_LINE_PATTERNS = {
    "engine_config": "Initializing a V1 LLM engine",
    "non_default_args": "non-default args",
    "attention_backend": "attention backend",
    "kv_cache_size": "GPU KV cache size",
    "max_concurrency": "Maximum concurrency for",
    "kv_cache_memory": "Available KV cache memory",
    "model_load": "Model loading took",
    "ple_lower": "ple",
    "offload": "offload",
    "cuda_oom": "CUDA out of memory",
    "torch_oom": "OutOfMemoryError",
    "cuda_error": "CUDA error",
    "illegal_memory": "illegal memory access",
    "engine_dead": "EngineDeadError",
    "traceback": "Traceback (most recent call last)",
}


def capture_engine_lines(log_path, max_hits=4):
    lines = tail_log_lines(log_path, max_bytes=4_000_000)
    found = {}
    for key, needle in ENGINE_LINE_PATTERNS.items():
        if key == "ple_lower":
            hits = [ln.strip()[:1200] for ln in lines
                    if ("ple" in ln.lower() and ("offload" in ln.lower() or "PLE" in ln))]
        else:
            hits = [ln.strip()[:1200] for ln in lines if needle in ln]
        if hits:
            found[key] = {"count": len(hits), "lines": hits[:max_hits]}
    backend = None
    for ln in lines:
        m = re.search(r"Using (\S+) attention backend", ln)
        if m and backend is None:
            backend = m.group(1)
    found["attention_backend_name"] = backend
    found["oomish"] = bool(found.get("cuda_oom") or found.get("torch_oom"))
    return found


def print_engine_lines(tag, found):
    print(f"flashnext-smoke: [{tag}] engine lines:", flush=True)
    for key in ("engine_config", "attention_backend", "kv_cache_size", "max_concurrency",
                "kv_cache_memory", "model_load", "ple_lower", "offload"):
        for ln in (found.get(key) or {}).get("lines", [])[:2]:
            print(f"  {key}: {ln[:1100]}", flush=True)
    print(f"  attention backend : {found.get('attention_backend_name')}", flush=True)
    for key in ("cuda_oom", "torch_oom", "cuda_error", "illegal_memory", "engine_dead"):
        for ln in (found.get(key) or {}).get("lines", [])[:2]:
            print(f"  !! {key}: {ln[:600]}", flush=True)


def stop_server(reason):
    print(f"flashnext-smoke: stopping vLLM ({reason})", flush=True)
    proc = CURRENT_SERVER.get("proc")
    if proc is not None and proc.poll() is None:
        try:
            os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        except Exception:
            pass
    subprocess.run(["pkill", "-TERM", "-f", "vllm.entrypoints"], check=False)
    deadline = time.time() + 90
    while time.time() < deadline and vllm_procs():
        time.sleep(3)
    if vllm_procs():
        if proc is not None and proc.poll() is None:
            try:
                os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
            except Exception:
                pass
        subprocess.run(["pkill", "-9", "-f", "vllm.entrypoints"], check=False)
        time.sleep(10)
    deadline = time.time() + 240
    while time.time() < deadline:
        sample = gpu_sample()
        if sample is not None and sample["mem_mib"] < 8000:
            break
        time.sleep(5)
    CURRENT_SERVER["proc"] = None
    print(f"flashnext-smoke: server stopped, gpu={gpu_sample()}", flush=True)


def served_ids():
    try:
        return [m.get("id") for m in http_json(VLLM_API + "/models", timeout=5).get("data", [])]
    except Exception:
        return None


def start_server(flags, tag, timeout_s=1800):
    """Boot the pinned vLLM with the given full argv in THEIR serving_env().
    1800 s readiness = their wait_server + VLLM_PLE_OFFLOAD_READY_TIMEOUT
    number. Raises on death/timeout with the log tail printed; the boot is
    recorded under RESULTS['boots'][tag] either way."""
    log_path = WORKING_DIR / f"vllm-{tag}.log"
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", *flags]
    print(f"flashnext-smoke: starting vLLM [{tag}] (elapsed {elapsed_min():.1f} min):",
          " ".join(cmd), flush=True)
    handle = log_path.open("w", encoding="utf-8")
    t0 = time.time()
    proc = subprocess.Popen(cmd, env=serving_env(), stdout=handle, stderr=subprocess.STDOUT,
                            text=True, start_new_session=True)
    CURRENT_SERVER.update({"proc": proc, "log": str(log_path), "tag": tag, "flags": flags})
    boot = {"tag": tag, "flags": flags, "started_utc": datetime.utcnow().isoformat() + "Z"}
    RESULTS["boots"][tag] = boot
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if proc.poll() is not None:
            lines = tail_log_lines(log_path)
            print(f"flashnext-smoke: [{tag}] BOOT FAILURE rc={proc.returncode} — log tail:", flush=True)
            print("\n".join(lines[-150:]), flush=True)
            boot.update({"ok": False, "rc": proc.returncode, "boot_s": round(time.time() - t0, 1),
                         "log_tail": lines[-60:], "engine_lines": capture_engine_lines(log_path)})
            print_engine_lines(tag, boot["engine_lines"])
            save_results()
            raise RuntimeError(f"vLLM [{tag}] died during startup rc={proc.returncode}")
        if server_alive():
            ids = served_ids()
            found = capture_engine_lines(log_path)
            boot.update({"ok": True, "boot_s": round(time.time() - t0, 1), "served_ids": ids,
                         "engine_lines": found, "gpu_after": gpu_sample(),
                         "host_after": host_snapshot(f"after_boot_{tag}")})
            print(f"flashnext-smoke: vLLM ready [{tag}] in {boot['boot_s']} s, served ids {ids}",
                  flush=True)
            print_engine_lines(tag, found)
            print(f"flashnext-smoke: [{tag}] gpu after boot {boot['gpu_after']} | "
                  f"host {boot['host_after']}", flush=True)
            if ids != [QWEN_SERVED_MODEL_NAME]:
                print(f"flashnext-smoke: WARNING served ids {ids} != [{QWEN_SERVED_MODEL_NAME}]",
                      flush=True)
            save_results()
            return flags
        time.sleep(10)
    lines = tail_log_lines(log_path)
    print(f"flashnext-smoke: [{tag}] BOOT TIMEOUT — log tail:", flush=True)
    print("\n".join(lines[-150:]), flush=True)
    boot.update({"ok": False, "rc": None, "timeout": True, "boot_s": round(time.time() - t0, 1),
                 "log_tail": lines[-60:], "engine_lines": capture_engine_lines(log_path)})
    save_results()
    raise TimeoutError(f"vLLM [{tag}] not ready in {timeout_s}s")


def reset_prefix_cache():
    # vLLM API server: POST /reset_prefix_cache. Best effort — recorded, never fatal.
    try:
        req = urllib.request.Request(VLLM_ROOT + "/reset_prefix_cache", data=b"", method="POST")
        with urllib.request.urlopen(req, timeout=30) as resp:
            return resp.status
    except Exception as exc:
        return ("failed: " + repr(exc))[:120]




In [ ]:
# ===================== Phase 2 — BOOT (their argv verbatim; pre-registered retry rung) =====================
BOOT_TAG = None
try:
    if not RUNTIME_OK:
        raise RuntimeError("assemble failed — boot skipped")
    for _tag, _flags, _arch in BOOT_LADDER:
        try:
            ARCH_OVERRIDE["value"] = _arch
            print(f"flashnext-smoke: rung [{_tag}] TORCH_CUDA_ARCH_LIST={_arch!r}", flush=True)
            start_server(_flags, tag=_tag)
            BOOT_TAG = _tag
            break
        except Exception as _exc:
            traceback.print_exc()
            RESULTS["boots"].setdefault(_tag, {})["error"] = repr(_exc)[:600]
            print(f"flashnext-smoke: boot [{_tag}] FAILED — next rung of the ladder", flush=True)
            save_results()
            stop_server(f"cleanup after failed {_tag}")
    if BOOT_TAG is None:
        RESULTS["verdicts"]["boot"] = "BOOT FAILED on all rungs (device-detect arch, 12.0a, reduced)"
    else:
        _b = RESULTS["boots"][BOOT_TAG]
        _el = _b.get("engine_lines") or {}
        RESULTS["verdicts"]["boot"] = (
            f"BOOT OK on rung '{BOOT_TAG}'"
            + (" (DEGRADED-CONFIG: their exact config did not serve)" if BOOT_TAG == "reduced" else "")
            + f" in {_b.get('boot_s')} s — attention backend {_el.get('attention_backend_name')}, "
            + f"gpu after boot {_b.get('gpu_after')}")
except Exception as _exc:
    traceback.print_exc()
    RESULTS["verdicts"].setdefault("boot", "BOOT PHASE ERROR: " + repr(_exc)[:300])
print("flashnext-smoke: BOOT VERDICT:", RESULTS["verdicts"].get("boot"), flush=True)
save_results()


# The smoke is unreadable without a live server — die loudly here rather than
# spend three hours playing games against a dead endpoint.
if BOOT_TAG is None:
    raise RuntimeError("Flash-Next server did not boot on any rung — smoke aborted")


In [ ]:
# ============ duck analyzer env — the ONLY wiring change vs the 27B kernel ============
# Reproduces the 27B bundle's exported setup_env EXACTLY (same keys,
# same stock sampling: temp 0.6 / top-p 0.95 / top-k 20 / thinking on) except:
#   * base URL + model id -> the Flash-Next server booted above (port 1234,
#     served name from their FLASH_SERVE_FLAGS);
#   * LOCAL_ANALYZER_CONTEXT_WINDOW 24576 + LOCAL_ANALYZER_MAX_OUTPUT 4096:
#     their server is --max-model-len 32768; the duck's default 32768 window
#     with no max_tokens could push prompt+completion past the server cap.
#     (The 27B baseline ran a 32768 window on a 65536-ctx server. Noted as a
#     deviation in the README; the stock 27B effective history is 4-9 turns.)
#   * PYTHONPATH NOT exported: the cu130 serving runtime must not shadow the
#     notebook's harness deps. The harness talks plain HTTP via `requests`;
#     only the vLLM server subprocess sees the extracted site-packages.
# These exports MUST land before the deploy pkls are loaded two cells down:
# tool_agent reads CONTEXT_WINDOW/MAX_OUTPUT at import time.
if BOOT_TAG is None:
    raise RuntimeError("no Flash-Next server — cannot wire the analyzer")
_booted_flags = CURRENT_SERVER["flags"]
_booted_ctx = int(_booted_flags[_booted_flags.index("--max-model-len") + 1])
if _booted_ctx >= 32768:
    _ctx_window, _max_out = "24576", "4096"
else:
    # reduced rung (16k server): shrink the window proportionally
    _ctx_window, _max_out = "12288", "2048"

duck_env = {
    "USE_TF": "0",
    "TRANSFORMERS_NO_TF": "1",
    "TRANSFORMERS_NO_TORCHVISION": "1",
    "VLLM_NO_USAGE_STATS": "1",
    "LOCAL_ANALYZER_BASE_URL": VLLM_API,
    "OPENAI_BASE_URL": VLLM_API,
    "LOCAL_ANALYZER_PROVIDER": "vllm",
    "OPENAI_PROVIDER": "vllm",
    "LOCAL_ANALYZER_MODEL_ID": QWEN_SERVED_MODEL_NAME,
    "INFERENCE_ANALYZER_MODEL": QWEN_SERVED_MODEL_NAME,
    "LOCAL_ANALYZER_APP_NAME": "ARC3 Agent Harness",
    "LOCAL_ANALYZER_CONTEXT_WINDOW": _ctx_window,
    "LOCAL_ANALYZER_MAX_OUTPUT": _max_out,
    "LOCAL_ANALYZER_TOOL_STEPS": "0",
    "LOCAL_ANALYZER_TOOL_TIMEOUT": "30",
    "LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS": "1024",
    "LOCAL_ANALYZER_YIELD_SECONDS": "60",
    "LOCAL_ANALYZER_TEMPERATURE": "0.6",
    "LOCAL_ANALYZER_TOP_P": "0.95",
    "LOCAL_ANALYZER_TOP_K": "20",
    "LOCAL_ANALYZER_ENABLE_THINKING": "true",
    "MULTIMODAL_CONTEXT": "current_grid",
    "MULTIMODAL_UPSCALE": "4",
}
os.environ.update(duck_env)
SETUP_ENV_PATH.write_text(json.dumps(duck_env, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _run_shell_commands(filename, *, label, check):
    # Teardown shim: the v12 run cell calls this with teardown_commands.json,
    # which targets the 27B bundle's own vLLM pid file. This kernel's server
    # is ours — stop it directly and flush the results sink.
    print(f"taaf.kaggle: {label} ({filename}) -> flashnext stop_server", flush=True)
    try:
        stop_server(label)
    except Exception as exc:  # noqa: BLE001
        print("flashnext-smoke: teardown stop failed:", repr(exc)[:300], flush=True)
    save_results()


assert os.environ.get("INFERENCE_ANALYZER_MODEL") == QWEN_SERVED_MODEL_NAME
RESULTS["meta"]["duck_env"] = {k: v for k, v in duck_env.items()
                               if k.startswith(("LOCAL_", "INFERENCE_", "OPENAI_", "MULTIMODAL_"))}
save_results()
print("\n✅ duck analyzer wired to Flash-Next")
print("Analyzer endpoint:", os.environ["LOCAL_ANALYZER_BASE_URL"])
print("Analyzer model:   ", os.environ["INFERENCE_ANALYZER_MODEL"])
print("Context window:   ", _ctx_window, "| max output:", _max_out,
      "| server max-model-len:", _booted_ctx)


In [ ]:
# Boot attestation (flashnext variant of doctrine v2, 2026-08-17): the
# assembled model view must be the RadixArk Flash-Next NVFP4 package and the
# LIVE server must answer through the duck's analyzer endpoint. Discriminators
# from the model card + the gate v4 Kaggle run: architectures
# Qwen4ExpForConditionalGeneration, model_type qwen4_exp, 206 safetensors
# > 186 GB (NVFP4 experts + BF16-converted PLE tables; the PLE conversion is
# hash-verified against FLASHNEXT_GCP_MODEL_INFO.json in the assemble cell).
# A wrong view must DIE here, before any game action is spent.
import hashlib as _hashlib
import urllib.request as _rq

_cfg_path = QWEN_MODEL_PATH / "config.json"
_cfg_raw = _cfg_path.read_bytes()
_cfg = json.loads(_cfg_raw)
assert _cfg.get("architectures") == ["Qwen4ExpForConditionalGeneration"], (
    f"attest FAIL: architectures {_cfg.get('architectures')}")
assert _cfg.get("model_type") == "qwen4_exp", (
    f"attest FAIL: model_type {_cfg.get('model_type')}")
print("attest: config sha256", _hashlib.sha256(_cfg_raw).hexdigest())

_shards = sorted(QWEN_MODEL_PATH.glob("*.safetensors"))
_total = sum(p.stat().st_size for p in _shards)
print(f"attest: {len(_shards)} shards, {_total} bytes total")
assert len(_shards) == 206, f"attest FAIL: shard count {len(_shards)} != 206"
assert _total > 186_000_000_000, f"attest FAIL: total shard bytes {_total} too small"

# Greedy decode fingerprint through the analyzer endpoint — logged (not
# asserted) for cross-run comparison, and proves the served model answers.
_base = (os.environ.get("LOCAL_ANALYZER_BASE_URL") or "http://127.0.0.1:1234/v1").rstrip("/")
if not _base.endswith("/v1"):
    _base += "/v1"
_body = json.dumps({
    "model": os.environ["INFERENCE_ANALYZER_MODEL"],
    "messages": [{"role": "user", "content": "Reply with exactly the sum of 17 and 25, then the word quack."}],
    "temperature": 0.0,
    "max_tokens": 48,
    "chat_template_kwargs": {"enable_thinking": False, "preserve_thinking": True},
}).encode()
_req = _rq.Request(_base + "/chat/completions", data=_body, headers={
    "Content-Type": "application/json",
    "Authorization": "Bearer " + (os.environ.get("LOCAL_ANALYZER_API_KEY") or "EMPTY"),
})
with _rq.urlopen(_req, timeout=300) as _resp:
    _reply = json.loads(_resp.read())["choices"][0]["message"].get("content") or ""
print("attest: decode fingerprint", repr(_reply)[:160])
print("attest: decode sha256", _hashlib.sha256(_reply.encode()).hexdigest())
print("attest: OK — Flash-Next NVFP4 signature + live analyzer endpoint verified before any game")


In [ ]:
def _soft_end_time(max_runtime_s: float, *, run_as_submission: bool) -> datetime | None:
    if run_as_submission or max_runtime_s <= 0:
        return None
    budget = max(1.0, max_runtime_s)
    buffer = min(SOFT_DEADLINE_BUFFER_S, budget / 2)
    start = datetime.fromtimestamp(NOTEBOOK_START_EPOCH)
    return start + timedelta(seconds=budget - buffer)


def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ.get("ARC_BASE_URL", "http://gateway:8001/"),
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


@contextlib.contextmanager
def _tee_to_file(log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_file = open(log_path, "w", buffering=1)
    original_stdout = sys.stdout
    original_stderr = sys.stderr
    sys.stdout = _Tee(original_stdout, log_file)
    sys.stderr = _Tee(original_stderr, log_file)
    try:
        yield
    finally:
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        log_file.close()


class _Tee:
    def __init__(self, *streams: TextIO) -> None:
        self._streams = streams

    def write(self, data: str) -> int:
        n = 0
        for stream in self._streams:
            n = stream.write(data)
        return n

    def flush(self) -> None:
        for stream in self._streams:
            stream.flush()

    def isatty(self) -> bool:
        return any(getattr(stream, "isatty", lambda: False)() for stream in self._streams)

In [ ]:
true_submission = _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
run_as_submission = _env_bool("TAAF_RUN_AS_SUBMISSION", False) or true_submission
os.environ["ONLY_RESET_LEVELS"] = "true"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if run_as_submission else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if run_as_submission else "0"

with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = run_as_submission
target.is_competition_rerun = true_submission
soft_end = _soft_end_time(float(getattr(target, "max_runtime_s", 0.0) or 0.0), run_as_submission=run_as_submission)

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [ ]:
# Smoke/eval hook: THE MODEL-SWAP READ. A NORMAL COMMIT runs the STOCK duck
# on all 25 public games against the Flash-Next server, eval geometry
# (7920 s per game, concurrency 28 from the serialized solver). One
# phase, no grafts, ONLY_RESET_LEVELS=true, stock sampling. The scored rerun
# path (KAGGLE_IS_COMPETITION_RERUN) never enters this branch.
GAMES_25 = [
    "ar25-0c556536",
    "bp35-0a0ad940",
    "cd82-fb555c5d",
    "cn04-2fe56bfb",
    "dc22-fdcac232",
    "ft09-0d8bbf25",
    "g50t-5849a774",
    "ka59-38d34dbb",
    "lf52-271a04aa",
    "lp85-305b61c3",
    "ls20-9607627b",
    "m0r0-492f87ba",
    "r11l-495a7899",
    "re86-8af5384d",
    "s5i5-18d95033",
    "sb26-7fbdac44",
    "sc25-635fd71a",
    "sk48-d8078629",
    "sp80-589a99af",
    "su15-1944f8ab",
    "tn36-ef4dde99",
    "tr87-cd924810",
    "tu93-0768757b",
    "vc33-5430563c",
    "wa30-ee6fef47"
]
SMOKE_PHASES = [
    ("flashnext", GAMES_25, 7920),
]
FN_PHASE_ERRORS = []
FN_ALL_RUNS = []

if not run_as_submission:
    import arc_agi
    from taaf.game_api import ArcadeSpec, GameAPI

    def _resolve_env_dir():
        candidates = [
            Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"),
            Path("/kaggle/input/arc-prize-2026-arc-agi-3/environment_files"),
        ]
        for cand in candidates:
            if cand.is_dir():
                return str(cand)
        for hit in Path("/kaggle/input").rglob("environment_files"):
            if hit.is_dir():
                return str(hit)
        raise RuntimeError("environment_files dir not found in /kaggle/input")

    _env_dir = _resolve_env_dir()
    _spec = ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=_env_dir)
    bm.games = [GameAPI(env_name=name, arcade_spec=_spec) for name in SMOKE_PHASES[0][1]]
    bm.n_passes = 1
    bm.game_weights = None
    bm.label = "flashnext-smoke-" + SMOKE_PHASES[0][0]
    bm.solver.max_runtime_s_per_game = float(SMOKE_PHASES[0][2])
    soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=11400)
    print(f"smoke hook: phases={[(p[0], len(p[1]), p[2]) for p in SMOKE_PHASES]} "
          f"env_dir={_env_dir} concurrency={bm.solver.concurrency} "
          f"per_game_cap={bm.solver.max_runtime_s_per_game}s soft_end={soft_end}")
else:
    print("scored rerun: smoke hook inert — full competition games")

print("Benchmark analyzer model:", os.environ.get("INFERENCE_ANALYZER_MODEL"))


In [ ]:
# ---- flashnext smoke telemetry: THE READ. Per-game actions / levels / score /
# turns / tokens from the framework mirror + the ToolAgent session counters,
# vLLM /metrics deltas (prompt + generation tokens, prefix-cache queries/hits,
# preemptions), and a 60 s queue sampler: 28 client workers vs their
# --max-num-seqs 22 means requests queue by design — running/waiting/kv are
# part of the record.
import json as _tel_json
import re as _tel_re
import threading as _tel_threading
import time as _tel_time
import urllib.request as _tel_rq

from inference.framework import solver as _solver_mod

_tel_lock = _tel_threading.Lock()
_TEL_PATH = WORKING_DIR / "flashnext_smoke_results.json"
FN_SESSIONS = {}      # phase -> {game_id: {...}} filled at session exit
FN_PHASES = []        # ordered phase records
_FN_CURRENT = {"phase": None, "t0": None, "metrics0": None,
               "sampler_stop": None, "queue_samples": None}


def _metrics_url():
    base = (os.environ.get("LOCAL_ANALYZER_BASE_URL") or "http://127.0.0.1:1234/v1").rstrip("/")
    if base.endswith("/v1"):
        base = base[:-3]
    return base + "/metrics"


_COUNTER_KEYS = (
    "vllm:prompt_tokens_total", "vllm:generation_tokens_total",
    "vllm:prefix_cache_queries_total", "vllm:prefix_cache_hits_total",
    "vllm:request_success_total", "vllm:num_preemptions_total",
)
_GAUGE_KEYS = (
    "vllm:num_requests_running", "vllm:num_requests_waiting",
    "vllm:kv_cache_usage_perc",
)


def _fn_scrape(keys):
    out = {}
    try:
        with _tel_rq.urlopen(_metrics_url(), timeout=20) as resp:
            text = resp.read().decode("utf-8", errors="replace")
    except Exception as exc:  # noqa: BLE001
        return {"error": repr(exc)}
    for line in text.splitlines():
        if not line or line.startswith("#"):
            continue
        for key in keys:
            # exact metric name only ("...waiting" must not match "...waiting_by_reason")
            if line.startswith(key + "{") or line.startswith(key + " "):
                m = _tel_re.match(r"^\S+(?:\{[^}]*\})?\s+([0-9.eE+-]+)", line)
                if m:
                    try:
                        out[key] = out.get(key, 0.0) + float(m.group(1))
                    except ValueError:
                        pass
    return out


def _fn_phase_begin(phase):
    stop = _tel_threading.Event()
    samples = []

    def _queue_sampler():
        while not stop.is_set():
            g = _fn_scrape(_GAUGE_KEYS)
            if "error" not in g:
                g["t"] = round(_tel_time.time(), 1)
                samples.append(g)
            stop.wait(60)

    with _tel_lock:
        _FN_CURRENT.update({"phase": phase, "t0": _tel_time.monotonic(),
                            "metrics0": _fn_scrape(_COUNTER_KEYS),
                            "sampler_stop": stop, "queue_samples": samples})
        FN_SESSIONS.setdefault(phase, {})
    _tel_threading.Thread(target=_queue_sampler, daemon=True).start()
    print(f"[fn-tel] phase {phase} begin metrics0={_FN_CURRENT['metrics0']}", flush=True)


def _fn_phase_end(phase, game_runs):
    stop = _FN_CURRENT.get("sampler_stop")
    if stop is not None:
        stop.set()
    m1 = _fn_scrape(_COUNTER_KEYS)
    m0 = _FN_CURRENT.get("metrics0") or {}
    delta = {k: (m1.get(k, 0.0) - m0.get(k, 0.0)) for k in _COUNTER_KEYS if k in m1}
    wall = _tel_time.monotonic() - (_FN_CURRENT.get("t0") or _tel_time.monotonic())
    games = []
    for game_run in game_runs:
        apl = list(game_run.actions_per_level or [])
        actions = sum(apl) if apl else len(game_run.history)
        sess = FN_SESSIONS.get(phase, {}).get(game_run.game_id, {})
        games.append({
            "game_id": game_run.game_id,
            "state": game_run.state,
            "levels_completed": game_run.levels_completed,
            "number_of_levels": game_run.number_of_levels,
            "final_score": game_run.final_score,
            "actions": actions,
            "actions_per_level": apl,
            "base_actions_per_level": list(game_run.base_actions_per_level or []),
            "wallclock_s": game_run.final_wallclock_seconds,
            "solver_note": game_run.solver_note,
            "turns": sess.get("turns"),
            "session_total_tokens": sess.get("total_tokens"),
            "session_generated_tokens": sess.get("generated_tokens"),
            "history_messages_at_exit": sess.get("history_messages"),
            "context_budget_tokens": sess.get("context_budget_tokens"),
            "yield_seconds": sess.get("yield_seconds"),
            "tool_steps": sess.get("tool_steps"),
        })
    n = max(1, len(games))
    queries = delta.get("vllm:prefix_cache_queries_total", 0.0)
    hits = delta.get("vllm:prefix_cache_hits_total", 0.0)
    gen = delta.get("vllm:generation_tokens_total", 0.0)
    prompt = delta.get("vllm:prompt_tokens_total", 0.0)
    q = list(_FN_CURRENT.get("queue_samples") or [])
    running = [s["vllm:num_requests_running"] for s in q if "vllm:num_requests_running" in s]
    waiting = [s["vllm:num_requests_waiting"] for s in q if "vllm:num_requests_waiting" in s]
    kv = [s["vllm:kv_cache_usage_perc"] for s in q if "vllm:kv_cache_usage_perc" in s]
    rec = {
        "phase": phase,
        "harness": "stock-duck",
        "server_tag": CURRENT_SERVER["tag"],
        "analyzer_env": {k: v for k, v in os.environ.items()
                         if k.startswith(("LOCAL_ANALYZER", "INFERENCE_ANALYZER", "MULTIMODAL"))},
        "wall_s": round(wall, 1),
        "games": games,
        "n_games": len(games),
        "mean_actions": round(sum(g["actions"] for g in games) / n, 2),
        "mean_levels": round(sum(g["levels_completed"] for g in games) / n, 3),
        "mean_score": round(sum(float(g["final_score"] or 0.0) for g in games) / n, 4),
        "zero_level_games": sum(1 for g in games if g["levels_completed"] == 0),
        "mean_turns": round(sum(float(g["turns"] or 0) for g in games) / n, 1),
        "metrics_delta": delta,
        "prefix_hit_rate": round(hits / queries, 4) if queries else None,
        "prefill_per_gen_token": round(prompt / gen, 2) if gen else None,
        "gen_tok_s": round(gen / wall, 1) if wall else None,
        "queue": {
            "n_samples": len(q),
            "running_mean": round(sum(running) / len(running), 1) if running else None,
            "running_max": max(running) if running else None,
            "waiting_mean": round(sum(waiting) / len(waiting), 1) if waiting else None,
            "waiting_max": max(waiting) if waiting else None,
            "kv_usage_mean": round(sum(kv) / len(kv), 3) if kv else None,
            "kv_usage_max": round(max(kv), 3) if kv else None,
        },
    }
    with _tel_lock:
        FN_PHASES.append(rec)
    try:
        _TEL_PATH.write_text(_tel_json.dumps(
            {"phases": FN_PHASES, "phase_errors": FN_PHASE_ERRORS}, indent=1, default=str),
            encoding="utf-8")
    except Exception:  # noqa: BLE001
        pass
    print(f"[fn-tel] phase {phase} end: games={rec['n_games']} mean_actions={rec['mean_actions']} "
          f"mean_levels={rec['mean_levels']} mean_score={rec['mean_score']} "
          f"zero_level={rec['zero_level_games']} turns={rec['mean_turns']} "
          f"prefix_hit={rec['prefix_hit_rate']} prefill/gen={rec['prefill_per_gen_token']} "
          f"gen_tok_s={rec['gen_tok_s']} queue={rec['queue']} wall={rec['wall_s']}s", flush=True)


# Session exit seam: record turns + token counters per game for the phase.
_inner_play = _solver_mod._HarnessGameSession.play


def _tel_play(self):
    try:
        return _inner_play(self)
    finally:
        try:
            gid = getattr(getattr(self.game, "game_run", None), "game_id", "?")
            an = self.analyzer
            rec = {
                "turns": int(getattr(self, "analysis_step", 0) or 0),
                "total_tokens": int(getattr(an, "_session_total_tokens", 0) or 0),
                "generated_tokens": int(getattr(an, "_session_generated_tokens", 0) or 0),
                "history_messages": len(getattr(an, "_history_messages", []) or []),
                "context_budget_tokens": getattr(an, "_context_budget_tokens", None),
                "yield_seconds": getattr(an, "_yield_seconds", None),
                "tool_steps": getattr(an, "_tool_steps", None),
            }
            with _tel_lock:
                FN_SESSIONS.setdefault(_FN_CURRENT.get("phase") or "?", {})[gid] = rec
        except Exception:  # noqa: BLE001
            pass


if not getattr(_solver_mod._HarnessGameSession.play, "_fn_tel", False):
    _tel_play._fn_tel = True
    _solver_mod._HarnessGameSession.play = _tel_play

print("[fn-tel] installed; metrics url =", _metrics_url(), flush=True)


In [ ]:
run_context = contextlib.nullcontext() if run_as_submission else _tee_to_file(WORKING_DIR / "stdout.log")
with run_context:
    preamble = (BUNDLE_DIR / "preamble.txt").read_text(encoding="utf-8")
    print(preamble)
    print(f"deploy.kaggle: working_dir             = {WORKING_DIR}")
    print(f"deploy.kaggle: run_as_submission       = {run_as_submission}")
    print(f"deploy.kaggle: competition_rerun       = {true_submission}")
    print(f"deploy.kaggle: soft_end_time           = {soft_end}")
    print("---")

    bundled_git_status = BUNDLE_DIR / "git_status.txt"
    if bundled_git_status.is_file():
        (WORKING_DIR / "git_status.txt").write_text(
            bundled_git_status.read_text(encoding="utf-8"),
            encoding="utf-8",
        )

    if true_submission:
        # Competition reruns use Kaggle's live gateway instead of the bundled offline games.
        os.environ.setdefault("ARC_API_KEY", "test-key-123")
        os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
        os.environ.setdefault("SCHEME", "http")
        os.environ.setdefault("HOST", "gateway")
        os.environ.setdefault("PORT", "8001")
        os.environ.setdefault("OPERATION_MODE", "competition")
        os.environ.setdefault("ENVIRONMENTS_DIR", "")
        os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

        deadline = time.monotonic() + 600.0
        last_error = ""
        while time.monotonic() < deadline:
            try:
                with urlopen("http://gateway:8001/api/games", timeout=10) as response:
                    if response.status < 500:
                        break
            except Exception as exc:
                last_error = repr(exc)
            time.sleep(5)
        else:
            raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")

        bm.games = _competition_games()
        bm.n_passes = 1
        bm.game_weights = None

    try:
        # flashnext smoke: ONE phase — the STOCK duck against the Flash-Next
        # server. The scored-rerun path takes exactly one pass with the
        # competition games (and would need the 27B kernel, not this one).
        for _phase_name, _phase_games, _phase_cap in SMOKE_PHASES:
            if not run_as_submission:
                FN_ALL_RUNS.extend(bm.game_runs)
                bm.game_runs = []
                bm.games = [GameAPI(env_name=_n, arcade_spec=_spec) for _n in _phase_games]
                bm.n_passes = 1
                bm.game_weights = None
                bm.label = "flashnext-smoke-" + _phase_name
                bm.solver.max_runtime_s_per_game = float(_phase_cap)
                print(f"=== PHASE {_phase_name}: {len(_phase_games)} games "
                      f"per_game_cap={_phase_cap}s concurrency={bm.solver.concurrency} "
                      f"model={os.environ.get('INFERENCE_ANALYZER_MODEL')} ===", flush=True)
            _fn_phase_begin(_phase_name)
            try:
                await bm.run(
                    soft_end_time=soft_end,
                    runtime_environment=target,
                    minimal_diagnostics=run_as_submission,
                )
            except Exception as _phase_exc:  # noqa: BLE001
                import traceback

                FN_PHASE_ERRORS.append(f"{_phase_name}: {type(_phase_exc).__name__}: {_phase_exc}")
                print(f"PHASE {_phase_name} RAISED {type(_phase_exc).__name__}: {_phase_exc}",
                      flush=True)
                traceback.print_exc()
            try:
                _fn_phase_end(_phase_name, list(bm.game_runs))
            except Exception:  # noqa: BLE001
                pass
            if run_as_submission:
                break
        if not true_submission and Path("/kaggle/input").exists():
            try:
                import pandas as pd

                submission = pd.DataFrame(
                    data=[["1_0", "1", True, 1]],
                    columns=["row_id", "game_id", "end_of_game", "score"],
                )
                submission.to_parquet(WORKING_DIR / "submission.parquet", index=False)
            except Exception as exc:
                print(f"taaf.kaggle: could not write offline dummy submission: {exc!r}", flush=True)
    finally:
        _run_shell_commands("teardown_commands.json", label="teardown", check=False)

In [ ]:
# ---- flashnext smoke final report (grep for FLASHNEXT SMOKE / READ) ----
BASELINE_27B_STOCK = {
    "source": "pooled stock phases of arc3-tp-smoke / arc3-tp1b-smoke / arc3-tp1c-smoke "
              "(27B-FP8, same GPU class, same geometry, 2026-08-29)",
    "levels_per_game": "1.00 / 1.04 / 0.84-0.88",
    "zero_level_games": "8-9 of 25",
    "mean_local_score": "3.5-4.9",
}
print("=" * 78)
print("FLASHNEXT SMOKE RESULTS — STOCK duck x Flash-Next NVFP4 (model-swap read)")
print("boot:", RESULTS["verdicts"].get("boot"))
print("27B stock baseline:", BASELINE_27B_STOCK)
by = {p["phase"]: p for p in FN_PHASES}
p = by.get("flashnext")
verdict = "UNREADABLE"
detail = ""
if not p or not p["n_games"]:
    print("PHASE flashnext: MISSING")
else:
    print(f"PHASE flashnext: games={p['n_games']} mean_actions={p['mean_actions']} "
          f"mean_levels={p['mean_levels']} mean_score={p['mean_score']} "
          f"zero_level={p['zero_level_games']} mean_turns={p['mean_turns']} "
          f"prefix_hit={p['prefix_hit_rate']} prefill/gen={p['prefill_per_gen_token']} "
          f"gen_tok_s={p['gen_tok_s']} wall={p['wall_s']}s")
    print(f"queue: {p['queue']}")
    for g in sorted(p["games"], key=lambda g: g["game_id"]):
        print(f"  {g['game_id']}: levels={g['levels_completed']}/{g['number_of_levels']} "
              f"score={g['final_score']} actions={g['actions']} turns={g['turns']} "
              f"gen_tok={g['session_generated_tokens']} state={g['state']}")
    lev = p["mean_levels"]
    zero = p["zero_level_games"]
    detail = (f"levels {lev:.3f} (27B stock 1.00/1.04/0.84-0.88) zero_level {zero}/25 "
              f"(27B 8-9/25) actions {p['mean_actions']} score {p['mean_score']}")
    # Pre-registered rules (README.md, fixed before flight):
    if lev >= 1.3 or (zero <= 6 and lev >= 1.0):
        verdict = "PASS"
    elif lev < 0.9:
        verdict = "FAIL"
    else:
        verdict = "INCONCLUSIVE"
print(f"FLASHNEXT SMOKE READ: {verdict} ({detail}) phase_errors={FN_PHASE_ERRORS}")
results = {
    "kernel": "arc3-flashnext-smoke",
    "harness": "stock-duck",
    "boot": RESULTS["verdicts"].get("boot"),
    "baseline_27b_stock": BASELINE_27B_STOCK,
    "phases": FN_PHASES,
    "phase_errors": FN_PHASE_ERRORS,
    "verdict": verdict,
    "detail": detail,
}
(WORKING_DIR / "flashnext_smoke_results.json").write_text(
    json.dumps(results, indent=1, default=str), encoding="utf-8")
print("wrote", WORKING_DIR / "flashnext_smoke_results.json")
